# Data Preprocessing Pipeline

This notebook runs the Mahmut preprocessing pipeline for the banking intent dataset: normalization, near-duplicate detection, stratified splitting, and the final data-funnel summary.

In [5]:
import importlib
import sys

sys.path.insert(0, '..')

import pandas as pd
import src.preprocess as preprocess

importlib.reload(preprocess)
from src.preprocess import normalize_df, remove_near_duplicates, save_splits, save_data_funnel

raw_df = pd.read_csv('../data/raw/synthetic_raw.csv')
print(f'Raw rows: {len(raw_df)}')


Raw rows: 3010


## 1) Normalize the text column

We apply the reusable text normalization step before any similarity checks or train/test splitting. This preserves punctuation and symbols while cleaning whitespace and non-ASCII quote/dash variants.

In [6]:
normalized_df = normalize_df(raw_df)
print(f'Rows after normalization: {len(normalized_df)}')
normalized_df.head()

Rows after normalization: 3010


,text,intent
0,why does my recent transactions list stop at l...,latest_transactions
1,Cancel my Harbourline Insurance direct debit.,direct_debit
2,Got a notice from Southgate Energy saying $214...,pay_bill
3,hi um so I'm heading to Bali for a semester ex...,abroad
4,My new card hasnt arived yet.,card_issues


## 2) Near-duplicate detection

We compute sentence embeddings and remove same-intent pairs above the similarity threshold of 0.95, while saving different-intent high-similarity pairs for manual review.

In [7]:
cleaned_df = remove_near_duplicates(
    normalized_df,
    processed_path='../data/processed/synthetic_clean.csv',
    results_dir='../results',
)
print(f'Rows after near-duplicate removal: {len(cleaned_df)}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Top 20 most similar pairs:
 left_index  right_index  similarity  same_intent        intent_left       intent_right
         73         2026    0.974495         True             freeze             freeze
       1840         1850    0.964101         True       direct_debit       direct_debit
       1004         1011    0.958960         True            balance            balance
        961         1002    0.948597         True            balance            balance
       2072         2175    0.946676         True             freeze             freeze
       1820         1840    0.945807         True       direct_debit       direct_debit
        815          910    0.944769         True          atm_limit          atm_limit
       1003         1008    0.942359         True            balance            balance
        126         2455    0.939411         True      joint_account      joint_account
        135         1347    0.937140         True      business_loan      business_loan
     

## 3) Stratified train/validation/test split

We keep the original class balance across the three splits using a fixed random seed of 42.

In [8]:
train_df, val_df, test_df = save_splits(
    cleaned_df,
    train_path='../data/processed/train.csv',
    val_path='../data/processed/val.csv',
    test_path='../data/processed/test.csv',
)


train split intent counts:
intent
abroad                 151
address                150
app_error              150
atm_limit              151
balance                150
business_loan          150
card_issues            150
cash_deposit           150
direct_debit           150
freeze                 150
high_value_payment     150
joint_account          151
latest_transactions    151
pay_bill               150
Name: count, dtype: int64

val split intent counts:
intent
abroad                 32
address                33
app_error              32
atm_limit              32
balance                32
business_loan          32
card_issues            33
cash_deposit           32
direct_debit           32
freeze                 32
high_value_payment     33
joint_account          32
latest_transactions    32
pay_bill               32
Name: count, dtype: int64

test split intent counts:
intent
abroad                 32
address                32
app_error              33
atm_limit              32


## 4) Data funnel summary

This final table records the number of rows after each preprocessing stage so that we can document the data flow clearly.

In [9]:
funnel_df = save_data_funnel(
    raw_df,
    normalized_df,
    cleaned_df,
    train_df,
    val_df,
    test_df,
    output_path='../results/data_funnel.csv',
)
funnel_df

Saved data funnel to ../results/data_funnel.csv
                       stage  rows
                         raw  3010
         after_normalization  3010
after_near_duplicate_removal  3007
                       train  2104
                         val   451
                        test   452


,stage,rows
0,raw,3010
1,after_normalization,3010
2,after_near_duplicate_removal,3007
3,train,2104
4,val,451
5,test,452
